# 9.4 Stage 4: Return to Initial Pose (Expanded)

Returns arm to initial pose smoothly.

---

```python id="d5c3o2"
elif t < 5*d:
    self.log_stage(4, "Returning to initial pose")

    r = (t - 3*d) / (2*d)
    for i, j in enumerate(self.joints):
        self.low_cmd.motor_cmd[j].q = self.interp(self.target_extend[i], self.initial_pose[i], r)
        self.low_cmd.motor_cmd[j].dq = 0
        self.low_cmd.motor_cmd[j].kp = self.kp
        self.low_cmd.motor_cmd[j].kd = self.kd
        self.low_cmd.motor_cmd[j].tau = 0
```

---

## 🧠 Big Picture: What Is This Stage Doing?

This stage tells the robot:

```text
"Return smoothly from the current extended pose back to the original starting pose"
```

---

## 🔄 Trajectory Reversal

This is the **first time you reverse a trajectory**.

---

### Compare with previous stages:

| Stage   | Motion                         |
| ------- | ------------------------------ |
| Stage 2 | initial_pose → target_raise    |
| Stage 3 | target_raise → target_extend   |
| Stage 4 | target_extend → initial_pose ✅ |

---

### 🧠 Key Insight

```text
You are closing the motion loop
```

---

## ⏱️ When Does This Stage Run?

```python
elif t < 5*d:
```

---

### Time window:

```text
3d → 5d
```

If `d = 3s`:

```text
9s → 15s
```

---

## ⚠️ Important Design Choice

### This stage lasts **2× longer** than others

---

### Why?

Look at interpolation:

```python
r = (t - 3*d) / (2*d)
```

---

### Compare durations:

| Stage   | Duration |
| ------- | -------- |
| Stage 2 | d        |
| Stage 3 | d        |
| Stage 4 | 2d ✅     |

---

### 🧠 Interpretation

```text
Return motion is slower and more controlled
```

---

## 🎯 Step 1: Compute Interpolation Ratio

```python
r = (t - 3*d) / (2*d)
```

---

### Behavior:

| Time   | r   |
| ------ | --- |
| t = 3d | 0   |
| t = 4d | 0.5 |
| t = 5d | 1   |

---

### Key difference:

```text
r increases HALF as fast compared to earlier stages
```

---

### Result:

> 🧠 **Slower motion → smoother and safer return**

---

## 🎯 Step 2: Reverse the Interpolation

```python
self.interp(self.target_extend[i], self.initial_pose[i], r)
```

---

### Compare:

| Stage   | Interpolation      |
| ------- | ------------------ |
| Stage 3 | raise → extend     |
| Stage 4 | extend → initial ✅ |

---

### Meaning:

```text
Move backward along the trajectory
```

---

## 🧠 Why Return to `initial_pose`?

Because:

```text
initial_pose = actual measured starting configuration
```

---

### Benefits:

* Guarantees safe final posture
* Matches real robot configuration
* Avoids drift

---

## ⚙️ PD Control (Same as Before)

```python
dq = 0
kp = self.kp
kd = self.kd
tau = 0
```

---

### Behavior:

* Position error drives motion
* Damping ensures stability

---

## 🔄 Motion Behavior

Over time:

```text
r increases slowly
→ q_target moves toward initial_pose
→ torque applied
→ joints move back
```

---

### Result:

```text
Smooth, controlled return motion
```

---

## ⚠️ Why Make Return Slower?

This is a **real robotics design decision**.

---

### Reasons:

### 1. Safety

```text
Returning too fast → instability
```

---

### 2. Gravity Effects

When lowering the arm:

```text
Gravity assists motion → risk of overshoot
```

---

### 3. Energy Dissipation

Slower motion allows:

```text
Better damping → less oscillation
```

---

### 4. Mechanical Stress

```text
Fast reversal → high torque spikes
```

---

## 🔬 Control Theory Insight

This stage demonstrates:

> **Time scaling of trajectories**

---

### Same path, different timing:

```text
Fast → aggressive motion  
Slow → smooth motion
```

---

## 🧠 System-Level Interpretation

You now have:

```text
Full motion cycle:
Start → Raise → Extend → Return
```

---

### This is a **closed-loop behavior**

---

## 🤖 RL Interpretation

This stage represents:

```text
Returning to a "home state"
```

---

### In RL:

* Often needed for:

  * episode reset
  * safe termination
  * repeatable experiments

---

### Equivalent:

```python
state → action → ... → return_to_start
```

---

## 🔄 Behavior Composition Insight

This stage shows:

> Behaviors are not just forward actions—they include **recovery and reset**

---

## ⚠️ What If You Skipped This Stage?

Then:

* Robot stays in extended pose
* No clean reset
* Next run may start from bad state

---

## 🔄 Analogy

Think of this like:

```text
You raise your arm → open hand → bring arm back down slowly
```

---

You don’t:

* snap it back instantly

You:

* control the descent

---

## 🔥 Hidden Engineering Insight

This stage demonstrates:

> **Reversibility + temporal shaping of motion**

---

Which is critical in:

* manipulation tasks
* locomotion cycles
* RL episodic control

---

## 🚀 Summary

This stage:

| Step                  | Role               |
| --------------------- | ------------------ |
| Compute `r`           | Slower progression |
| Reverse interpolation | Return path        |
| Apply PD control      | Stable motion      |
| Extend duration       | Increase safety    |

---

### Result:

```text
Arm smoothly returns to original pose over a longer, safer trajectory
```

---

> 🔥 This stage teaches **trajectory reversal and timing control**, both essential for real-world robotics.

